# 08 · Uncertainty, Resolution, and Synthetic Tests

*Ask what the data resolve, not merely what the inversion returned.*

**Learning objectives**

- derive posterior covariance and model resolution
- interpret off-diagonal covariance and resolution kernels
- design honest checkerboard and spike tests
- propagate slip uncertainty into seismic moment

**Prerequisites:** Chapters 03–04  
**Estimated time:** 60–90 minutes

Notation follows the [glossary](../docs/glossary.md); axes, units, signs, and
ordering follow [GeoDef conventions](../docs/conventions.md).


## Motivation

A colorful slip map is an estimate, not an image of the fault. We need separate
answers to two questions: how broadly would repeated noisy data move the
estimate, and how does the inverse operator spatially average the true model?
Uncertainty and resolution answer those questions under the assumed model.


## Theory deepening: two different model-space questions

Under the linear-Gaussian assumptions,

$$C_m=(G^TWG+\lambda L^TL)^{-1}.$$

Its diagonal gives marginal variances; off-diagonal terms reveal trade-offs,
often including anticorrelation between neighboring smoothed patches. The
resolution matrix satisfies

$$E[\hat m]=Rm_{true},\qquad
R=(G^TWG+\lambda L^TL)^{-1}G^TWG.$$

Each row of $R$ is an averaging kernel. The diagonal alone cannot show where
leaked amplitude came from. Checkerboards test one chosen wavelength; spikes
test localization. Neither validates geometry, covariance, or linearity.
Moment uncertainty follows from the linear functional
$M_0=\mu\,a^Tm$: $\sigma_{M_0}^2=\mu^2a^TC_ma$.


## Posterior covariance and resolution

For the linear-Gaussian problem, the posterior **model covariance** is

$$ C_m = \left(G^{\mathsf T} W G + \lambda L^{\mathsf T} L\right)^{-1}. $$

Its diagonal gives the per-patch variance; $\sqrt{\operatorname{diag} C_m}$ is
the 1-$\sigma$ slip uncertainty.

The **model resolution matrix** relates the recovered model to the truth,

$$ \hat{\mathbf m} = R\,\mathbf m_{\text{true}}, \qquad R = G^{-g} G, $$

where $G^{-g}$ is the generalized inverse. Each recovered patch is a *weighted
average* of the true slip — the corresponding **row of $R$** is that patch's
resolution kernel. If $R \approx I$, slip is perfectly resolved; broad rows mean
the estimate is a blurred average of its neighbours.

There is an unavoidable **trade-off**: strong smoothing lowers uncertainty but
smears resolution; weak smoothing sharpens resolution but inflates uncertainty.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.matrix(
    fault, geodef.data.gnss(lon=glon, lat=glat, east=_zero, north=_zero, up=_zero, sigma_east=_one, sigma_north=_one, sigma_up=_one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.data.gnss(
    lon=glon, lat=glat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_sta, sigma), sigma_north=np.full(n_sta, sigma), sigma_up=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

In [ ]:
result = geodef.solve(fault, gnss, components="dip",
                       regularization="laplacian", regularization_strength=1.0)

## Per-patch uncertainty

`model_uncertainty` returns the 1-$\sigma$ slip error for each patch. Deep,
poorly-sampled patches carry the largest uncertainty.

In [ ]:
unc = geodef.invert.model_uncertainty(result, fault, gnss)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
geodef.plot.slip(fault, result.dip_slip, ax=ax1, title="Recovered slip",
                 colorbar_label="Slip (m)")
geodef.plot.uncertainty(fault, unc, ax=ax2, title="1-sigma uncertainty")
plt.tight_layout()

## Resolution

`model_resolution` returns the diagonal of $R$. Values near 1 mark
well-resolved patches (typically shallow, beneath the station network); values
near 0 mark patches the data barely see.

In [ ]:
R = geodef.invert.model_resolution(result, fault, gnss)   # full (N, N) matrix
res_diag = np.diag(R)                              # per-patch resolution
geodef.plot.resolution(fault, res_diag, vmin=0, vmax=1,
                       title="Model resolution (diagonal of R)")
print(f"resolution range: {res_diag.min():.2f} - {res_diag.max():.2f}")

## Checkerboard test

A classic recoverability check: put a *known* checkerboard of slip on the fault,
generate synthetic data, invert, and see where the checkers survive. Sharp
checkers = well resolved; smeared or missing checkers = poorly resolved.

In [ ]:
checker = 1.0 + 0.9 * ((-1.0) ** i) * ((-1.0) ** j)     # alternating high/low
m_check = np.concatenate([np.zeros(N), checker])
d_check = G_full @ m_check + rng.normal(0.0, sigma, G_full.shape[0])
gnss_check = geodef.data.gnss(lon=glon, lat=glat, east=d_check[0::3], north=d_check[1::3], up=d_check[2::3],
                         sigma_east=np.full(n_sta, sigma), sigma_north=np.full(n_sta, sigma), sigma_up=np.full(n_sta, sigma))
rec = geodef.solve(fault, gnss_check, components="dip",
                    regularization="laplacian", regularization_strength=0.3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
geodef.plot.slip(fault, checker, ax=ax1, title="Input checkerboard",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, rec.dip_slip, ax=ax2, title="Recovered",
                 colorbar_label="Slip (m)")
plt.tight_layout()

## Moment magnitude with uncertainty

The scalar seismic moment $M_0 = \mu \sum_k s_k A_k$ is linear in slip, so its
uncertainty propagates from the slip covariance. Here we approximate it by
sampling slip from the posterior.

In [ ]:
Cm = geodef.invert.model_covariance(result, fault, gnss)
samples = rng.multivariate_normal(result.dip_slip, Cm, size=500)
# each sample is a length-N per-patch slip; magnitude() takes per-patch slip.
mw = np.array([fault.magnitude(np.clip(s, 0, None)) for s in samples])
print(f"Mw = {mw.mean():.2f} +/- {mw.std():.2f}")

## Checkpoint questions

**Can a parameter have low uncertainty and low resolution?**

<details><summary>Answer</summary>



Yes. A strong prior can make it precise while the estimate is a biased spatial average.



</details>

**What does one row of $R$ show?**

<details><summary>Answer</summary>



How every true parameter contributes to one recovered parameter.



</details>

**What does a checkerboard fail to test?**

<details><summary>Answer</summary>



Untried wavelengths, geometry error, covariance error, and nonlinear model discrepancy.



</details>


## Common mistakes

- Plotting only `diag(R)` hides spatial leakage.
- Calling conditional covariance total uncertainty ignores geometry and model error.
- Choosing a checker size after seeing the recovery turns assessment into presentation.


## Recap

- Covariance quantifies spread; resolution quantifies averaging.
- Regularization couples their trade-off through filtered modes.
- Synthetic tests answer only the specific recovery question posed.

Return to the [workflow decision guides](../docs/workflow.md) when adapting
this method. The course map in [the tutorial README](README.md) identifies the
next chapter and optional branches.


## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are published separately in `solutions/08_uncertainty_and_resolution_solutions.ipynb`.

1. Run two checker wavelengths at two strengths and tabulate recovery.
2. Move a spike down dip until it is no longer localized.
3. Propagate covariance to $M_0$ and $M_w$.
4. Challenge: approach $R=I$ by reducing noise and regularization in a full-rank small system.


## Further reading

- Backus & Gilbert (1968), resolution and averaging kernels.
- Menke (2018), covariance and model resolution.
- Funning et al. (2005), geodetic resolution testing.
